# Capture: временная таблица внутри типового метода ЗУП

Остановим один вызов дважды, посмотрим стек и временную таблицу, затем изменим запрос только внутри выбранного вызова.

Пример рассчитан на ЗУП КОРП 3.1.38.92 и платформу 8.5.1.1529; дата демоснимка — 01.08.2021. Используйте отдельную демо-копию и соответствующую ей выгрузку исходников. Подготовка окружения — в [README](README.md). Сеанс закрывается последней ячейкой; при прерывании выполните `runtime.close()`.

## Подготовка

Используйте API wheels 0.1.17. Перед запуском измените `PLATFORM_BIN`, `CONNECTION_STRING` и `SOURCE_ROOT` в следующей ячейке. Имя пользователя уже задано; пароль пустой.

In [ ]:
from IPython.display import display
from onec_runtime.config import RuntimeConfig
from onec_runtime.session import ExtensionMode, RuntimeSessionConfig
from onec_runtime_jupyter import InteractiveRuntimeSession

PLATFORM_BIN = r'C:\Program Files\1cv8\8.5.1.1529\bin'
CONNECTION_STRING = r'File="C:\demo\ЗУП";'
SOURCE_ROOT = r'C:\exports\ЗУП'
EXTENSION_MODE = ExtensionMode.AUTO  # MANUAL для ИБ с заранее установленным расширением.

runtime = InteractiveRuntimeSession.start(
    RuntimeSessionConfig(
        runtime=RuntimeConfig(
            platform_bin=PLATFORM_BIN,
            connection_string=CONNECTION_STRING,
            username='Савинская З.Ю. (Системный программист)',
        ),
        source_root=SOURCE_ROOT,
        extension_mode=EXTENSION_MODE,
    )
)

## Выбираем сотрудников

Берём тех же работающих на дату сотрудников, что и в обзоре. Для `КадровыеДанныеСотрудников` нужен список ссылок.

In [ ]:
%%bsl
ДатаФОТ = Дата(2021, 8, 1);
ПараметрыСотрудников = КадровыйУчет.ПараметрыПолученияСотрудниковОрганизацийПоСпискуФизическихЛиц();
ПараметрыСотрудников.НачалоПериода = ДатаФОТ;
ПараметрыСотрудников.ОкончаниеПериода = ДатаФОТ;
ПараметрыСотрудников.РаботникиПоТрудовымДоговорам = Истина;

СписокСотрудников = КадровыйУчет.СотрудникиОрганизации(
    Истина, ПараметрыСотрудников).Скопировать(, "Сотрудник");
СписокСотрудников.Свернуть("Сотрудник");
СписокСотрудников.Колонки.Добавить("Период", Новый ОписаниеТипов("Дата"));
Для Каждого СтрокаСотрудника Из СписокСотрудников Цикл
    СтрокаСотрудника.Период = ДатаФОТ;
КонецЦикла;

In [ ]:
%%bsl
СотрудникиДемо = СписокСотрудников.ВыгрузитьКолонку("Сотрудник");

## Две точки внутри типового метода

A — перед выгрузкой итогового запроса; B — перед возвратом таблицы. Следующая Python-ячейка находит эти инструкции в вашей выгрузке `КадровыйУчет` и устанавливает точки через путь к модулю и номер строки. Выгрузка должна соответствовать запущенной конфигурации.

In [ ]:
from pathlib import Path

MODULE_PATH = r'CommonModules\КадровыйУчет\Ext\Module.bsl'
lines = (Path(SOURCE_ROOT) / MODULE_PATH).read_text(encoding='utf-8-sig').splitlines()
starts = [index for index, line in enumerate(lines)
          if line.startswith('Функция КадровыеДанныеСотрудников(')]
if len(starts) != 1:
    raise ValueError('Проверьте выгрузку: метод КадровыеДанныеСотрудников не найден однозначно')
ends = [index for index in range(starts[0] + 1, len(lines))
        if lines[index].startswith('КонецФункции')]
if not ends:
    raise ValueError('Проверьте выгрузку: конец метода не найден')
end = ends[0]

def line_of(statement):
    found = [index + 1 for index in range(starts[0], end)
             if lines[index].strip() == statement]
    if len(found) != 1:
        raise ValueError(f'Проверьте исходный код метода: {statement}')
    return found[0]

CAPTURE_LINE_A = line_of('КадровыеДанныеСотрудников = Запрос.Выполнить().Выгрузить();')
CAPTURE_LINE_B = line_of('Возврат КадровыеДанныеСотрудников;')
runtime.add_capture_point(MODULE_PATH, CAPTURE_LINE_A)
runtime.add_capture_point(MODULE_PATH, CAPTURE_LINE_B)
print('Точки A/B:', CAPTURE_LINE_A, CAPTURE_LINE_B)

## Запускаем вызов

Вызов находится в отдельной BSL-ячейке. Она остановится в точке A; последующие ячейки работают с тем же вызовом.

In [ ]:
%%bsl
ПланПовтор = КадровыйУчет.КадровыеДанныеСотрудников(
    Истина, СотрудникиДемо,
    "ФОТ,Подразделение,ГоловнаяОрганизация,Организация", ДатаФОТ);
Результат = ПланПовтор.Итог("ФОТ");

In [ ]:
import pandas as pd

status_a = runtime.status()
assert status_a.state.value == 'captured'
stack = runtime.runtime_api.capture_stack(cursor=0, limit=20)
display(pd.DataFrame(stack['frames'])[['level', 'module_type', 'line']])
print('Кадров в стеке:', stack['total'])

## Читаем временную таблицу

`КонтекстОтладки` доступен, пока исходный вызов остановлен. Переносим в Python только три нужных столбца.

In [ ]:
%%bsl
ЗапросДемоВТ = Новый Запрос;
ЗапросДемоВТ.МенеджерВременныхТаблиц = КонтекстОтладки.Запрос.МенеджерВременныхТаблиц;
ЗапросДемоВТ.Текст = "ВЫБРАТЬ Сотрудник, Организация, ФОТ ИЗ ВТКадровыеДанныеСотрудников";
СнимокВТ = ЗапросДемоВТ.Выполнить().Выгрузить();

In [ ]:
temporary = СнимокВТ.to_df(refs='uuid')
display(temporary)
print('Строк:', len(temporary), 'ФОТ:', temporary['ФОТ'].sum(), '₽')

In [ ]:
resume_b = runtime.resume_capture()
assert resume_b.state.value == 'captured'
assert resume_b.operation_id == status_a.operation_id
print('Останов B:', resume_b.stop_sequence)

In [ ]:
%%bsl
СнимокВыхода = КонтекстОтладки.КадровыеДанныеСотрудников.Скопировать(
    , "Сотрудник,Организация,ФОТ");

In [ ]:
captured_output = СнимокВыхода.to_df(refs='uuid')
keys = ['Сотрудник', 'Организация']
sort_rows = lambda frame: frame.sort_values(keys).reset_index(drop=True)
pd.testing.assert_frame_equal(sort_rows(temporary), sort_rows(captured_output))
display(captured_output)

In [ ]:
completed = runtime.resume_capture()
assert completed.succeeded and completed.state.value == 'completed'
assert completed.operation_id == resume_b.operation_id
runtime.clear_capture_points()
print('Обычный результат:', completed.result, '₽')

## Меняем результат одного вызова

Повторно ставим те же точки. В останове A меняем текст локального `Запрос`: новый запрос читает прежнюю временную таблицу и умножает ФОТ на 1,1. Исходная конфигурация не меняется.

In [ ]:
runtime.add_capture_point(MODULE_PATH, CAPTURE_LINE_A)
runtime.add_capture_point(MODULE_PATH, CAPTURE_LINE_B)

In [ ]:
%%bsl
ПланПовтор = КадровыйУчет.КадровыеДанныеСотрудников(
    Истина, СотрудникиДемо,
    "ФОТ,Подразделение,ГоловнаяОрганизация,Организация", ДатаФОТ);
Результат = ПланПовтор.Итог("ФОТ");

In [ ]:
experiment_a = runtime.status()
assert experiment_a.state.value == 'captured'

In [ ]:
%%bsl
КонтекстОтладки.Запрос.Текст = "ВЫБРАТЬ Сотрудник, Организация, ФОТ * 1.1 КАК ФОТ ИЗ ВТКадровыеДанныеСотрудников";

In [ ]:
experiment_b = runtime.resume_capture()
assert experiment_b.state.value == 'captured'
assert experiment_b.operation_id == experiment_a.operation_id

In [ ]:
%%bsl
СнимокВыхода = КонтекстОтладки.КадровыеДанныеСотрудников.Скопировать(
    , "Сотрудник,Организация,ФОТ");

In [ ]:
from decimal import Decimal

changed_output = СнимокВыхода.to_df(refs='uuid')
expected = sort_rows(temporary).copy()
expected['ФОТ'] = expected['ФОТ'].map(lambda value: Decimal(str(value)) * Decimal('1.1'))
actual = sort_rows(changed_output)
pd.testing.assert_frame_equal(expected[keys], actual[keys])
assert [Decimal(str(value)) for value in expected['ФОТ']] == [
    Decimal(str(value)) for value in actual['ФОТ']]
display(changed_output)
print('ФОТ после изменения запроса:', changed_output['ФОТ'].sum(), '₽')

In [ ]:
experiment_done = runtime.resume_capture()
assert experiment_done.succeeded and experiment_done.state.value == 'completed'
assert experiment_done.operation_id == experiment_b.operation_id
runtime.clear_capture_points()

## Новый вызов снова использует типовой запрос

Изменение локального объекта действовало только в остановленном вызове. Повторим операцию без точек.

In [ ]:
%%bsl
ПланПовтор = КадровыйУчет.КадровыеДанныеСотрудников(
    Истина, СотрудникиДемо,
    "ФОТ,Подразделение,ГоловнаяОрганизация,Организация", ДатаФОТ);
Результат = ПланПовтор.Итог("ФОТ");

In [ ]:
%%bsl
ПланПовторКратко = ПланПовтор.Скопировать(, "Сотрудник,Организация,ФОТ");

In [ ]:
restored = ПланПовторКратко.to_df(refs='uuid')
pd.testing.assert_frame_equal(sort_rows(temporary), sort_rows(restored))
display(restored)
print('ФОТ после обычного вызова:', restored['ФОТ'].sum(), '₽')

## Граница опыта

Мы изменили свойство локального объекта `Запрос`, а не код уже начатого вызова. Для присваивания нового значения скалярной локальной переменной нужен другой механизм.

## Завершение

In [ ]:
runtime.close()
print('Сеанс закрыт')